# Bangla OCR Transformer Training Notebook

This notebook trains the Bangla OCR model using the training and validation images provided in the data folder.

## Dependencies

Before running this notebook, ensure you have the following packages installed:

```bash
pip install numpy torch torchvision matplotlib pillow mlconfig editdistance
```

You can also install PyTorch with CUDA support if you have a GPU available:

```bash
# For CUDA 11.8
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
```

**Note:** This notebook requires the preprocessed training and validation data to be available in the `data/Train` and `data/Validation` directories. If you haven't preprocessed the data yet, run the `preprocess.ipynb` notebook first.

## Import Required Libraries

In [ ]:
import numpy as np
import datetime, os, torch, time, pickle, sys, gc
from torch import nn
import torchvision.transforms as T
from torch.utils.data import Dataset
from pathlib import Path
import matplotlib.pyplot as plt

from services import Tokenizer, DataGenerator, LabelSmoothing
from model import BanglaOCR
from utils import findMaxTextLength, generatePlots

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Configuration

In [ ]:
# Training configuration
EPOCHS = 100
BATCH_SIZE = 4
HIDDEN_DIM = 256
NUM_DECODER_LAYERS = 4
NUM_HEADS = 4
LEARNING_RATE = 0.0002
WEIGHT_DECAY = 0.0004

print(f"Training Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Hidden Dimension: {HIDDEN_DIM}")
print(f"  Number of Decoder Layers: {NUM_DECODER_LAYERS}")
print(f"  Number of Heads: {NUM_HEADS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")

## Setup Paths and Load Character Sets

In [ ]:
# Get the current directory
parent = os.getcwd()

# Define paths
source_path_train = os.path.join(parent, 'data', 'Train', 'Images')
gt_path_train = os.path.join(parent, 'data', 'Train', 'Annotations')
source_path_val = os.path.join(parent, 'data', 'Validation', 'Images')
gt_path_val = os.path.join(parent, 'data', 'Validation', 'Annotations')
charset_path = os.path.join(parent, 'charset', 'printable1.txt')
punctuations_path = os.path.join(parent, 'charset', 'printable2.txt')

# Verify data directory exists
if not os.path.exists(source_path_train):
    raise FileNotFoundError(f"Training data folder not found at {source_path_train}")

if not os.path.exists(source_path_val):
    raise FileNotFoundError(f"Validation data folder not found at {source_path_val}")

print(f"Training data path: {source_path_train}")
print(f"Validation data path: {source_path_val}")

# Load character sets
charset_base = []
with open(charset_path, 'r', encoding='utf-8') as f:
    data = f.read()
    charset_base += data.split(',')
    charset_base = [x.strip() for x in charset_base]

with open(punctuations_path, 'r', encoding='utf-8') as f:
    data = f.read()
    charset_base += data.split(' ')
    charset_base = [x.strip() for x in charset_base]

charset_base.append(' ')

print(f"Character set size: {len(charset_base)}")

## Determine Maximum Text Length

In [ ]:
# Find maximum text length in the dataset
max_text_length = findMaxTextLength(gt_path_train, gt_path_val)
print(f"Maximum text length: {max_text_length}")

## Setup Device and Create Data Loaders

In [ ]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Create tokenizer
transform = T.Compose([T.ToTensor()])
tokenizer = Tokenizer(charset_base, max_text_length)
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    DataGenerator(source_path_train, gt_path_train, charset_base, max_text_length, transform),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    DataGenerator(source_path_val, gt_path_val, charset_base, max_text_length, transform),
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f"Number of batches in train loader: {len(train_loader)}")
print(f"Number of batches in validation loader: {len(val_loader)}")

## Create Model, Loss Function, and Optimizer

In [ ]:
# Create the model
model = BanglaOCR(
    vocab_len=tokenizer.vocab_size,
    max_text_length=max_text_length,
    hidden_dim=HIDDEN_DIM,
    nheads=NUM_HEADS,
    num_decoder_layers=NUM_DECODER_LAYERS
)
model = model.to(device)
print(f"Model created and moved to {device}")

# Count model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Create loss function and optimizer
criterion = LabelSmoothing(size=tokenizer.vocab_size, padding_idx=0, smoothing=0.1)
criterion = criterion.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

print("Loss function and optimizer created")

## Define Training and Validation Functions

In [ ]:
def train_epoch(model, criterion, optimizer, dataloader, tokenizer, device):
    """Train the model for one epoch"""
    model.train()
    total_loss = 0
    
    for imgs, labels_y in dataloader:
        imgs = imgs.to(device)
        labels_y = labels_y.to(device)
        
        optimizer.zero_grad()
        output = model(imgs.float(), labels_y.long()[:, :-1])
        
        loss = criterion(
            output.log_softmax(-1).contiguous().view(-1, tokenizer.vocab_size),
            labels_y[:, 1:].contiguous().view(-1).long()
        )
        
        loss.backward()
        total_loss += loss.item()
        optimizer.step()
        
        # Clean up memory
        del imgs, labels_y, output
        gc.collect()
        torch.cuda.empty_cache()
    
    return total_loss / len(dataloader)

def validate_epoch(model, criterion, dataloader, tokenizer, device):
    """Validate the model for one epoch"""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for imgs, labels_y in dataloader:
            imgs = imgs.to(device)
            labels_y = labels_y.to(device)
            
            output = model(imgs.float(), labels_y.long()[:, :-1])
            
            loss = criterion(
                output.log_softmax(-1).contiguous().view(-1, tokenizer.vocab_size),
                labels_y[:, 1:].contiguous().view(-1).long()
            )
            
            total_loss += loss.item()
            
            # Clean up memory
            del imgs, labels_y, output
            gc.collect()
            torch.cuda.empty_cache()
    
    return total_loss / len(dataloader)

def epoch_time(start_time, end_time):
    """Calculate elapsed time"""
    elapsed_time = end_time - start_time
    elapsed_hrs = int(elapsed_time / 3600)
    elapsed_mins = int((elapsed_time - elapsed_hrs * 3600) / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60 + elapsed_hrs * 3600))
    return elapsed_hrs, elapsed_mins, elapsed_secs

print("Training and validation functions defined")

## Training Loop

In [ ]:
# Initialize tracking variables
best_valid_loss = np.inf
train_loss_list = []
val_loss_list = []
counter = 0

print(f"Starting training for {EPOCHS} epochs...\n")
training_start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS} | Learning Rate: {scheduler.get_last_lr()}")
    
    epoch_start_time = time.time()
    
    # Train and validate
    train_loss = train_epoch(model, criterion, optimizer, train_loader, tokenizer, device)
    valid_loss = validate_epoch(model, criterion, val_loader, tokenizer, device)
    
    # Record losses
    train_loss_list.append(train_loss)
    val_loss_list.append(valid_loss)
    
    # Calculate time
    _, epoch_mins, epoch_secs = epoch_time(epoch_start_time, time.time())
    
    # Print progress
    print(f"Time: {epoch_mins}m {epoch_secs}s")
    print(f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")
    
    # Update best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print(f"✓ New best validation loss: {valid_loss:.4f}")
        counter = 0
    else:
        counter += 1
    
    # Adjust learning rate if needed
    if counter > 4:
        scheduler.step()
        counter = 0
        print("Learning rate adjusted")

training_end_time = time.time()
train_h, train_m, train_s = epoch_time(training_start_time, training_end_time)
print(f"\n{'='*60}")
print(f"Training completed!")
print(f"Total training time: {train_h}h {train_m}m {train_s}s")
print(f"Best validation loss: {best_valid_loss:.4f}")
print(f"{'='*60}")

## Save the Trained Model

In [ ]:
# Define model name and path
model_name = f"BanglaOCR_{EPOCHS}_{HIDDEN_DIM}_{NUM_HEADS}_{NUM_DECODER_LAYERS}"
model_path = os.path.join(parent, f"{model_name}.pt")

# Save the model using torch.save (saves as .pt file)
torch.save(model.state_dict(), model_path)

print(f"Model saved successfully to: {model_path}")
print(f"Model name: {model_name}")

# Also save tokenizer for future use
tokenizer_path = os.path.join(parent, 'tokenizer.pk')
pickle.dump(tokenizer, open(tokenizer_path, 'wb'))
print(f"Tokenizer saved to: {tokenizer_path}")

## Visualize Training Progress

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(12, 6))

min_val_loss = min(val_loss_list)
min_val_epoch = val_loss_list.index(min_val_loss)

plt.plot(range(1, len(train_loss_list) + 1), train_loss_list, color='blue', label='Train Loss', linewidth=2)
plt.plot(range(1, len(val_loss_list) + 1), val_loss_list, color='green', label='Validation Loss', linewidth=2)
plt.plot(min_val_epoch + 1, min_val_loss, marker='v', color='red', markersize=10, label=f'Best (Epoch {min_val_epoch + 1})')

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the plot
plot_path = os.path.join(parent, f'training_plot_{model_name}.png')
plt.savefig(plot_path, dpi=150)
print(f"Training plot saved to: {plot_path}")

plt.show()

## Training Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: {model_name}")
print(f"Total Epochs: {EPOCHS}")
print(f"Training Time: {train_h}h {train_m}m {train_s}s")
print(f"Best Validation Loss: {best_valid_loss:.4f} (Epoch {min_val_epoch + 1})")
print(f"Final Training Loss: {train_loss_list[-1]:.4f}")
print(f"Final Validation Loss: {val_loss_list[-1]:.4f}")
print(f"Model saved at: {model_path}")
print("="*60)